# Data cleaning, deduplication, and feature review

This notebook starts from the table written by notebook 1. It does not clean further, drop rows, or build new features. The point is to see what is already in `reports_level` before those steps.

The file is one row per case. `suspect_ingredients` and `reaction_pts` are pipe-separated lists on that row, not separate drug or reaction tables.


## 1. Load the starting table

Notebook 1 saves the shared file at `data/eda_start/reports_level.parquet`. Later notebooks should read that path rather than the XML zips.


In [ ]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 80)

REPORTS_LEVEL_PATH = Path("data/eda_start/reports_level.parquet")
if not REPORTS_LEVEL_PATH.exists():
    raise FileNotFoundError(
        "data/eda_start/reports_level.parquet is missing. "
        "Run notebook 1 first, then reopen this notebook."
    )

reports = pd.read_parquet(REPORTS_LEVEL_PATH)
print(f"loaded {REPORTS_LEVEL_PATH}")
print("shape:", reports.shape)


## 2. Grain

Each row should be one `safetyreportid`. A count above zero in the duplicate check means the latest-version filter in notebook 1 did not leave a unique case id.


In [2]:
print("columns:", list(reports.columns))
id_counts = reports["safetyreportid"].astype("string").value_counts(dropna=False)
n_duplicated_ids = int((id_counts > 1).sum())
n_rows_sharing_an_id = int(id_counts[id_counts > 1].sum())
print(f"unique safetyreportid: {reports['safetyreportid'].nunique(dropna=False):,}")
print(f"ids that appear more than once: {n_duplicated_ids:,}")
print(f"rows that share an id: {n_rows_sharing_an_id:,}")
display(reports.head(5))


columns: ['source_quarter', 'safetyreportversion', 'safetyreportid', 'primarysourcecountry', 'occurcountry', 'reporttype', 'serious', 'seriousnessdeath', 'seriousnesshospitalization', 'receivedate', 'receiptdate', 'fulfillexpeditecriteria', 'qualification', 'patientonsetage', 'patientonsetageunit', 'patientagegroup', 'patientweight', 'patientsex', 'age_years', 'patientweight_kg', 'receivedate_parsed', 'death', 'hospitalization', 'y_hosp_or_death', 'patientsex_label', 'qualification_label', 'n_drugs', 'n_suspect_drugs', 'suspect_ingredients', 'n_reactions', 'reaction_pts']
unique safetyreportid: 8,555,729
ids that appear more than once: 0
rows that share an id: 0


,source_quarter,safetyreportversion,safetyreportid,primarysourcecountry,occurcountry,reporttype,serious,seriousnessdeath,seriousnesshospitalization,receivedate,receiptdate,fulfillexpeditecriteria,qualification,patientonsetage,patientonsetageunit,patientagegroup,patientweight,patientsex,age_years,patientweight_kg,receivedate_parsed,death,hospitalization,y_hosp_or_death,patientsex_label,qualification_label,n_drugs,n_suspect_drugs,suspect_ingredients,n_reactions,reaction_pts
0,faers_xml_2020Q2,3,10004057,US,US,1,2,None,None,20140312,20200421,2,5,None,None,None,None,None,NaN,NaN,2014-03-12,False,False,False,None,consumer_or_non_health_professional,17,2,RUXOLITINIB,4,MALAISE | NAUSEA | HAEMOGLOBIN DECREASED | HEADACHE
1,faers_xml_2021Q2,7,10004234,FR,FR,2,1,None,None,20140312,20210629,1,1,81,801,6,64,None,81.0,64.00,2014-03-12,False,False,False,None,physician,8,1,FIBRINOGEN HUMAN\THROMBIN HUMAN,1,SKIN NECROSIS
2,faers_xml_2020Q4,3,10004657,US,US,1,1,None,None,20140312,20201103,1,4,71,801,None,81.63,2,71.0,81.63,2014-03-12,False,False,False,female,lawyer,2,1,ATORVASTATIN CALCIUM,1,TYPE 2 DIABETES MELLITUS
3,faers_xml_2020Q3,3,10004694,US,US,1,1,None,None,20140312,20200928,1,4,48,801,None,57,2,48.0,57.00,2014-03-12,False,False,False,female,lawyer,5,4,ATORVASTATIN CALCIUM,2,CATARACT | TYPE 2 DIABETES MELLITUS
4,faers_xml_2020Q4,2,10004696,PL,PL,1,1,None,None,20140312,20201021,1,3,23,801,None,None,1,23.0,NaN,2014-03-12,False,False,False,male,other_health_professional,5,2,SARCOSINE | QUETIAPINE,7,ENERGY INCREASED | TENSION | IRRITABILITY | LIBIDO INCREASED | ABNORMAL BEHA...


## 3. Coverage

`source_quarter` is the FAERS extract the kept version came from. `receivedate_parsed` is the FDA receive date, and only when that field had a full day.


In [3]:
display(reports["source_quarter"].value_counts(dropna=False).rename("n_cases").to_frame())
received = pd.to_datetime(reports["receivedate_parsed"], errors="coerce")
print("receivedate_parsed min:", received.min())
print("receivedate_parsed max:", received.max())
print(f"receivedate_parsed missing: {received.isna().mean():.1%}")


,n_cases
source_quarter,
faers_xml_2021Q3,425548
faers_xml_2026q2,422458
faers_xml_2022Q4,408336
faers_xml_2021Q2,398157
faers_xml_2022q1,394371
faers_xml_2025q3,391622
faers_xml_2022Q3,385886
faers_xml_2020Q1,383456
faers_xml_2021Q1,382947


receivedate_parsed min: 1986-12-24 00:00:00
receivedate_parsed max: 2026-06-30 00:00:00
receivedate_parsed missing: 0.0%


## 4. Missingness

This is the fraction of rows where each column is missing, highest first. Age, sex, weight, dates, ingredients, and reaction terms are the fields later cleaning will have to treat explicitly.


In [4]:
missing = reports.isna().mean().sort_values(ascending=False).rename("fraction_missing")
display(missing.to_frame().style.format("{:.1%}"))


,fraction_missing
patientweight,83.4%
patientweight_kg,83.4%
patientagegroup,74.1%
age_years,43.2%
patientonsetage,43.2%
patientonsetageunit,43.2%
seriousnessdeath,31.2%
seriousnesshospitalization,27.4%
patientsex,15.2%
patientsex_label,15.2%


## 5. Coded features

Sex, reporter type, and the seriousness flags are already mapped. The counts below are the values those columns actually hold. The numeric summary covers age, weight, and the drug and reaction counts stored on each case.


In [5]:
for column in [
    "patientsex_label",
    "qualification_label",
    "death",
    "hospitalization",
    "y_hosp_or_death",
]:
    print(column)
    display(reports[column].value_counts(dropna=False).rename("n_cases").to_frame())

display(
    reports[
        ["age_years", "patientweight_kg", "n_drugs", "n_suspect_drugs", "n_reactions"]
    ].describe()
)


patientsex_label


,n_cases
patientsex_label,
female,4267502
male,2975728
None,1302746
unknown,9753


qualification_label


,n_cases
qualification_label,
consumer_or_non_health_professional,3944479
other_health_professional,1869160
physician,1775545
pharmacist,500129
lawyer,346342
None,120074


death


...

hospitalization


,n_cases
hospitalization,
False,6907364
True,1648365


y_hosp_or_death


,n_cases
y_hosp_or_death,
False,6339693
True,2216036


,age_years,patientweight_kg,n_drugs,n_suspect_drugs,n_reactions
count,4.858392e+06,1.417272e+06,8555729.0,8555729.0,8555729.0
mean,5.520136e+01,7.497917e+01,4.024091,2.487204,3.031767
std,2.077359e+01,1.911411e+02,10.954676,8.798463,4.251131
min,0.000000e+00,0.000000e+00,1.0,0.0,1.0
25%,4.200000e+01,5.900000e+01,1.0,1.0,1.0
50%,5.900000e+01,7.227000e+01,2.0,1.0,2.0
75%,7.100000e+01,8.800000e+01,4.0,2.0,3.0
max,1.200000e+02,9.438000e+04,3902.0,2738.0,611.0


## 6. List fields

`suspect_ingredients` and `reaction_pts` use ` | ` between items. Empty means the case had no suspect ingredient, or no reaction term, after the light clean in notebook 1. The token counts describe those strings. They do not split them into new rows.


In [6]:
def token_count(series: pd.Series) -> pd.Series:
    text = series.astype("string").str.strip()
    empty = text.isna() | text.eq("")
    counts = text.str.split(r"\s\|\s").str.len()
    counts = counts.mask(empty, 0)
    return counts


for column in ["suspect_ingredients", "reaction_pts"]:
    counts = token_count(reports[column])
    empty_share = (counts == 0).mean()
    print(f"{column}: empty on {empty_share:.1%} of rows")
    print(f"{column}: token count describe")
    display(counts.describe().to_frame(column))


suspect_ingredients: empty on 0.3% of rows
suspect_ingredients: token count describe


,suspect_ingredients
count,8555729
unique,155
top,1
freq,6794183


reaction_pts: empty on 0.0% of rows
reaction_pts: token count describe


,reaction_pts
count,8555729
unique,204
top,1
freq,3539470


## 7. What this notebook does not do

Notebook 1 already dropped deleted case IDs and kept the latest `safetyreportversion` for each `safetyreportid`. This notebook only reviews that result.

Further deduplication, decisions about missing age or weight, and any new feature columns belong in later edits of this notebook. Do not treat the pipe-separated fields as a drug-level or reaction-level dataset until they are split on purpose.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=73434ccc-8a09-4578-aac6-0389f491d707' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>